3. 토큰 단위로 분할하기

tittoken

In [ ]:
# [목적] 여러 텍스트 분할 방식의 공통 입력으로 사용할 원본 문서를 UTF-8 형식으로 읽어옵니다.
# data 폴더의 키워드 문서를 문자열 변수 file에 저장합니다.
# 이후 예제들은 같은 file을 각기 다른 기준으로 나누어 결과를 비교합니다.
with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()

In [ ]:
# [목적] 텍스트 분할 전 원본 문서의 일부를 출력하여 내용을 확인합니다.
# 앞 500글자로 파일 읽기와 한글 인코딩이 정상인지 빠르게 점검합니다.
# 확인한 file 문자열은 아래 토큰 기반 분할 예제의 입력으로 사용됩니다.
print(file[:500])

In [ ]:
# [목적] tiktoken 토큰 수를 기준으로 문서를 분할하는 CharacterTextSplitter를 설정합니다.
# 이 예제는 모델이 실제로 처리하는 토큰 단위를 기준으로 조각 크기를 맞추는 방식입니다.
# 분할한 texts는 모델의 입력 길이를 관리하거나 임베딩·검색에 사용할 수 있습니다.
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    # 각 조각이 최대 300토큰이 되도록 설정합니다.
    chunk_size=300,
    # 조각 사이에 겹치는 토큰을 두지 않아 중복을 만들지 않습니다.
    chunk_overlap=0,
)

texts = text_splitter.split_text(file)

In [ ]:
# [목적] tiktoken 기준으로 나뉜 텍스트 조각의 개수를 확인합니다.
# texts는 앞 셀에서 만든 문자열 조각 목록이며, 길이는 문서가 몇 조각으로 나뉘었는지 뜻합니다.
# 이 수로 설정한 토큰 크기가 분할 결과에 미치는 영향을 확인할 수 있습니다.
print(len(texts))

TokenTextSplitter

In [ ]:
# [목적] TokenTextSplitter로 토큰 경계를 기준으로 문서를 분할하고 첫 결과를 확인합니다.
# 이 분할기는 텍스트를 모델용 토큰으로 계산해 지정한 크기마다 잘라, 입력 길이 제한을 다루는 데 적합합니다.
# 만들어진 texts는 긴 문서를 모델이나 검색 시스템에 전달하기 쉬운 크기로 나눈 결과입니다.
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(
    # 한 조각에 담을 최대 토큰 수입니다.
    chunk_size=300,
    # 조각을 겹치지 않게 만들어 중복 토큰을 제거합니다.
    chunk_overlap=0,
)

texts = text_splitter.split_text(file)
print(texts[0])

spaCy

In [ ]:
# [목적] spaCy 기반 분할 예제에 필요한 경고 설정과 SpacyTextSplitter를 준비합니다.
# spaCy는 문장 구조를 분석하는 라이브러리이며, SpacyTextSplitter는 그 분석 결과를 이용해 문장을 자연스럽게 나눕니다.
# 경고 메시지를 숨긴 뒤 다음 셀에서 분할기를 생성해 결과를 확인합니다.
import warnings
from langchain_text_splitters import SpacyTextSplitter

In [ ]:
# [목적] spaCy의 문장 분석을 이용해 텍스트를 자연스러운 문장 단위로 분할합니다.
# SpacyTextSplitter는 문장 경계를 우선 고려한 뒤, 설정한 최대 길이를 넘지 않도록 문서를 나눕니다.
# 분할 결과는 문장이 중간에 끊기는 일을 줄여 요약이나 검색 품질을 높이는 데 사용할 수 있습니다.
with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()

warnings.filterwarnings("ignore")

text_splitter = SpacyTextSplitter(
    # 한 조각의 최대 길이를 200자로 제한합니다.
    chunk_size=200,
    # 이웃한 조각에 50자를 반복해 경계 주변의 문맥을 유지합니다.
    chunk_overlap=50,
)

texts = text_splitter.split_text(file)
print(texts[0])

SentenceTransformers

In [ ]:
# [목적] SentenceTransformer 모델의 토큰 기준으로 문서를 분할하고 실제 토큰 수를 확인합니다.
# 이 예제는 임베딩 모델이 사용하는 토큰화 방식에 맞춰 조각 크기를 정하므로, 벡터 임베딩 입력을 안전하게 관리할 수 있습니다.
# 계산한 토큰 수와 text_chunks는 이후 임베딩 생성 또는 유사도 검색의 입력으로 활용할 수 있습니다.
from langchain_text_splitters import SentenceTransformersTokenTextSplitter

splitter = SentenceTransformersTokenTextSplitter(
    # 임베딩 모델에 전달할 각 조각의 최대 토큰 수입니다.
    chunk_size=200,
    # 조각 사이의 토큰 중복은 사용하지 않습니다.
    chunk_overlap=0
)

with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()

# SentenceTransformer가 텍스트 앞뒤에 자동으로 붙이는 시작·끝 토큰 수를 제외합니다.
count_start_and_stop_tokens = 2

# count_tokens로 모델 기준 토큰 수를 계산한 뒤, 자동 추가 토큰을 빼 실제 본문 토큰 수를 구합니다.
text_token_count = splitter.count_tokens(text=file) - count_start_and_stop_tokens
print(text_token_count)

text_chunks = splitter.split_text(text=file)
print(text_chunks[1])

NLTK

In [ ]:
# [목적] NLTK 문장 분할에 필요한 punkt 문장 인식 데이터를 내려받습니다.
# NLTK는 문장 끝을 판별하는 언어 처리 도구이며, punkt 데이터가 있어야 문장 경계를 찾을 수 있습니다.
# 내려받은 데이터는 다음 셀의 NLTKTextSplitter가 문서를 자연스럽게 나누는 데 사용합니다.
import nltk

nltk.download("punkt")

In [ ]:
# [목적] NLTKTextSplitter로 문장 경계를 기준으로 문서를 분할하고 첫 결과를 확인합니다.
# 이 분할기는 NLTK의 punkt 데이터를 이용해 문장이 끝나는 위치를 찾은 뒤, 지정한 크기 안에서 조각을 만듭니다.
# 문장 단위 조각은 질문 답변, 요약, 검색처럼 문맥이 중요한 작업에 활용하기 좋습니다.
from langchain_text_splitters import NLTKTextSplitter

with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()

text_splitter = NLTKTextSplitter(
    # 각 조각의 최대 길이를 200자로 제한합니다.
    chunk_size=200,
    # 조각 사이를 겹치지 않게 설정합니다.
    chunk_overlap=0,
)

texts = text_splitter.split_text(file)
print(texts[0])


KoNLPy

In [ ]:
# [목적] KoNLPy 기반 분할기로 한국어 텍스트를 문맥이 이어지도록 나눕니다.
# KonlpyTextSplitter는 한국어 처리 도구를 활용해 문서 경계를 고려하며, 긴 한국어 문서를 작은 조각으로 만듭니다.
# 만든 texts는 한국어 문서 검색이나 임베딩 같은 후속 LangChain 작업에 사용할 수 있습니다.
import chunk
from langchain_text_splitters import KonlpyTextSplitter

with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()
    
# 한 조각은 최대 200자로 만들고, 앞뒤 조각이 50자씩 겹치도록 설정합니다.
text_splitter = KonlpyTextSplitter(chunk_size=200, chunk_overlap=50)

texts = text_splitter.split_text(file)
print(texts[0])

Hugging Face 토크나이저

In [ ]:
# [목적] GPT-2 토크나이저로 계산한 토큰 수를 기준으로 텍스트를 분할합니다.
# 이 예제는 Hugging Face의 GPT-2 토크나이저를 LangChain 분할기에 연결해 특정 모델의 입력 단위에 맞추는 방식입니다.
# 이렇게 만든 texts는 GPT-2 계열 모델에 전달하거나 토큰 길이를 관리해야 하는 작업에 활용할 수 있습니다.
from transformers import GPT2TokenizerFast
from langchain_text_splitters import CharacterTextSplitter

# GPT-2 모델의 토크나이저를 불러옵니다.
hf_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# data/appendix-keywords.txt 파일을 열어서 f라는 파일 객체를 생성
with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()  # 파일의 내용을 읽어서 file 변수에 저장

text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    # 허깅페이스 토크나이저를 사용하여 CharacterTextSplitter 객체를 생성
    hf_tokenizer,
    chunk_size=300,
    chunk_overlap=50,
)

# state_of_the_union 텍스트를 분할하여 texts 변수에 저장
texts = text_splitter.split_text(file)
print(texts[1])  # texts 리스트의 1번째 요소를 출력